# 02 - Automatic differentiation and updates

**Goals:** compute gradients, observe accumulation, and take manual gradient descent steps.

The tensor operations from [01](01_tensors.ipynb) can also build a computation graph for differentiation. Run cells in order in a fresh kernel; each notebook supplies its own setup.

In [ ]:
import torch


## Differentiate a scalar

`requires_grad=True` asks autograd to track operations involving `x`. Before `backward()`, the directly created (leaf) tensor `x` has no stored gradient.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
print("y =", y.item(), "gradient before backward:", x.grad)


`backward()` applies the chain rule and stores the derivative in `x.grad`. Predict the result using $2x + 3$.

In [ ]:
y.backward()
print("dy/dx =", x.grad.item())  # 2 * 2 + 3 = 7


## Gradients accumulate

A new backward pass adds to the stored gradient. Recompute the expression to build a fresh graph; the previous graph is normally freed by `backward()`.

In [ ]:
(x**2 + 3 * x).backward()
print("after second backward:", x.grad.item())


Clear the gradient before computing a new update. Otherwise, old gradients contribute to later steps.

In [ ]:
x.grad.zero_()
print("after clearing:", x.grad.item())


## Differentiate a small linear model

This callable Python class computes $\mathbf{y} = \mathbf{x}W + \mathbf{b}$. Autograd tracks the tensor operations inside it without requiring a special model class.

In [ ]:
class Linear:
    def __init__(self, w, b):
        self.w = w
        self.b = b

    def __call__(self, x):
        return x @ self.w + self.b


For the values below:

$$
\begin{bmatrix} y_1 & y_2 \end{bmatrix}
=
\begin{bmatrix} 1 & 2 \end{bmatrix}
\begin{bmatrix} 3 & 4 \\ 5 & 6 \end{bmatrix}
+
\begin{bmatrix} 7 & 8 \end{bmatrix}
=
\begin{bmatrix} 20 & 24 \end{bmatrix}
$$

In [ ]:
inputs = torch.tensor([1.0, 2.0])  # Inputs do not need gradients here.
w = torch.tensor([[3.0, 4.0], [5.0, 6.0]], requires_grad=True)
b = torch.tensor([7.0, 8.0], requires_grad=True)

linear = Linear(w, b)
prediction = linear(inputs)
print("prediction:", prediction)


Compare the two outputs with a target and average their squared errors to get one scalar loss:

$$\mathrm{MSE}(\mathbf{y}, \mathbf{t}) = \frac{1}{n}\sum_{i=1}^{n}(y_i-t_i)^2.$$

Here $n=2$ counts output entries. A scalar loss gives `backward()` one objective to differentiate.

In [ ]:
target = torch.tensor([18.0, 22.0])
error = prediction - target
loss = (error ** 2).mean()
print("squared error loss:", loss.item())


The same backward call finds a gradient for every entry of `w` and `b`. Their gradient shapes match the parameter shapes; the inputs need no stored gradient to learn the parameters.

In [ ]:
loss.backward()
print("gradient of w:", w.grad)
print("gradient of b:", b.grad)


## Take one gradient descent step

For the simpler loss `(weight - 3)**2`, the derivative is `2 * (weight - 3)`. First calculate the loss and gradient at the starting value.

In [ ]:
weight = torch.tensor(0.0, requires_grad=True)
loss = (weight - 3)**2
loss.backward()
print("loss:", loss.item(), "gradient:", weight.grad.item())


Subtract the learning rate times the gradient. `no_grad()` keeps the parameter update out of the computation graph. The new weight is 0.6, closer to 3.

In [ ]:
with torch.no_grad():
    weight -= 0.1 * weight.grad
weight.grad.zero_()
print("updated weight:", weight.item())


## Repeat the update

Reset the weight to compare five steps from the same starting point. Each iteration builds a new graph. The printed loss is measured before that step updates the weight.

In [ ]:
weight = torch.tensor(0.0, requires_grad=True)
for step in range(5):
    loss = (weight - 3)**2
    loss.backward()
    with torch.no_grad():
        weight -= 0.1 * weight.grad
    weight.grad.zero_()
    print(f"step {step + 1}: weight={weight.item():.3f}, pre-update loss={loss.item():.3f}")


A gradient measures local sensitivity. Large steps can behave differently because the function curves; try the learning rates in the exercises.

## Keep values without the graph

`detach()` disconnects a tensor from its graph and may share its storage. This is useful when keeping tensor results without retaining their gradient history. Only leaf tensors retain `.grad` by default.

In [ ]:
a = torch.tensor([1., 2., 3.], requires_grad=True)
loss = (a**2).mean()
loss.backward()
print("gradient of mean square:", a.grad)
print("detached requires grad:", loss.detach().requires_grad)


Optional detail: differentiating a vector output directly requires a gradient argument of the same shape, forming a vector-Jacobian product. The scalar losses used here avoid needing that argument.

## Try it

1. Derive and check the gradient of `x**3 - 2*x` at `x=2`.

2. Remove gradient clearing from the manual loop; compare the path with the original.

3. Try learning rates 0.01, 0.5, and 1.1. Explain the behavior using the derivative.

**Next:** [Data and batches](03_data_and_batches.ipynb) organizes observations into the batches that will supply model inputs.